# 🧠 Build Your Own Jev – Part 3: Score (Rate on Scale)

**Series:** [Part 1: Noul](https://www.kaggle.com/code/syedasimalishah/build-your-own-jev-part-1-noul-yes-no) | [Part 2: Choice](https://www.kaggle.com/code/syedasimalishah/build-your-own-jev-part-2-choice-pick-one) | 👉 **Part 3: Score**

## What you will learn
How to make a model **rate something on your own scale**, like urgency from 0 to 3.
The score can fall between levels, like 2.2.

## The data
The same 6 support tickets from an online shoe store.

## What happens in this notebook
1. **Setup:** install the libraries, load the model, build the engine, and load the tickets.
2. **Build `score()`:** levels as digits (0, 1, 2...) → softmax → score = level × probability, added up.
3. **Two scales:** urgency (plain levels) and frustration (levels with examples).
4. **Priority:** mix the scores with customer data using your own weights → a sorted to-do list.
5. **Bonus, mini-Jev:** one `ask()` function that runs all 3 methods together.

## Before you run
- ⚙️ Settings → **Accelerator: GPU** and **Internet: On**
- Then click **Run All**
- Look for `🔧 CHANGE` in the code. It shows what you can edit for your own use.

In [1]:
import json, math, random
import torch
import pandas as pd
from transformers import AutoTokenizer, AutoModelForCausalLM

# 🔧 CHANGE: MODEL_NAME
#   "Qwen/Qwen2.5-0.5B-Instruct" → faster, less smart (good for testing)
#   "Qwen/Qwen2.5-3B-Instruct"   → smarter, slower, needs more GPU memory
MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"

device = "cuda" if torch.cuda.is_available() else "cpu"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.padding_side = "left"      # for batches: puts padding on the left, so every text ends at the last position
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# 🔧 CHANGE: dtype=torch.bfloat16 → uses half the memory (needs a newer GPU, like L4 or A100)
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, dtype=torch.float32).to(device)
model.eval()
print("Loaded", MODEL_NAME, "on", device)

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Loaded Qwen/Qwen2.5-1.5B-Instruct on cuda


In [2]:
# ===== THE ENGINE: all 3 methods are built on these small functions =====

# 🔧 CHANGE: SYSTEM_PROMPT → tell the model its job and its world.
#   Example: "You are a senior support agent at an online shoe store." → better domain judgment
SYSTEM_PROMPT = "You are a careful, fair judge. You answer with one short label only."

def format_state(state):
    """Text stays text. A dict becomes neat JSON, so the model sees every field and its name."""
    if isinstance(state, (dict, list)):
        return json.dumps(state, indent=2, ensure_ascii=False)
    return str(state)

def build_prompt(state, task):
    """Joins STATE + TASK (the question and the allowed answers) into the model's chat format."""
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": f"STATE:\n{format_state(state)}\n\n{task}"},
    ]
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

# 🔧 CHANGE: batch_size → bigger = faster. Make it smaller if you see "CUDA out of memory".
@torch.no_grad()
def next_token_logits(prompts, batch_size=8):
    """Runs the model on many prompts. Returns one row of logits per prompt: [n_prompts, vocab_size]."""
    rows = []
    for i in range(0, len(prompts), batch_size):
        batch = tokenizer(prompts[i:i + batch_size], return_tensors="pt", padding=True).to(device)
        out = model(**batch, logits_to_keep=1)   # keep only the LAST position (saves memory). Old transformers? Remove it.
        rows.append(out.logits[:, -1, :].float().cpu())
    return torch.cat(rows)

def token_ids(forms):
    """Token ids for each spelling, with and without a space.
    Example: ["Yes", "yes"] → ids of "Yes", " Yes", "yes", " yes"."""
    ids = set()
    for f in forms:
        for v in (f, " " + f):
            t = tokenizer.encode(v, add_special_tokens=False)
            if len(t) == 1:
                ids.add(t[0])
    if not ids:
        raise ValueError(f"None of {forms} is one token. Use short answers like letters or digits.")
    return sorted(ids)

def answer_logits(row, answers):
    """answers = {"yes": ["Yes", "yes"], "no": ["No", "no"]}
    Joins all spellings of one answer into ONE logit (logsumexp = adds their probabilities together)."""
    return torch.stack([torch.logsumexp(row[token_ids(forms)], dim=0) for forms in answers.values()])

def answer_probs(row, answers, temperature=1.0):
    """Keep only our answers → divide by temperature → softmax → probabilities that add up to 1."""
    return torch.softmax(answer_logits(row, answers) / temperature, dim=0)

def confidence(probs):
    """1.0 = all probability on one answer. 0.0 = spread evenly. Based on entropy."""
    if len(probs) < 2:
        return 1.0
    entropy = -(probs * probs.clamp_min(1e-12).log()).sum().item()
    return 1 - entropy / math.log(len(probs))

print("Engine ready ✅")

Engine ready ✅


In [3]:
# Support tickets from an online shoe store.
# Each ticket is ONE STATE: everything the model needs to judge it.
# 🔧 CHANGE: put your own data here (emails, reviews, chats, CVs...).
#   Keep only the fields that help the decision. Extra fields = more noise and slower runs.
TICKETS = [
    {
        "ticket_id": "T-101",
        "customer": {"tier": "gold", "orders_last_year": 14},
        "order": {"item": "Trail running shoes", "price_usd": 120, "status": "delivered", "days_since_delivery": 3},
        "message": "Wrong size AGAIN. I ordered 40 and got 42, same as last month. I don't want another exchange, just give me my money back.",
    },
    {
        "ticket_id": "T-102",
        "customer": {"tier": "standard", "orders_last_year": 1},
        "order": {"item": "Leather boots", "price_usd": 210, "status": "in_transit", "days_since_order": 16},
        "message": "It's been 16 days. Tracking hasn't moved in a week. If it doesn't arrive by Friday I'm reporting you to consumer protection.",
    },
    {
        "ticket_id": "T-103",
        "customer": {"tier": "standard", "orders_last_year": 3},
        "order": None,
        "message": "Checkout keeps failing with error 502 when I press Pay. Tried two browsers. My card is 4532 7781 0092 4421 if that helps.",
    },
    {
        "ticket_id": "T-104",
        "customer": {"tier": "new", "orders_last_year": 0},
        "order": None,
        "message": "Hi! I run a running club with 30 members. Do you give a discount if we order 30 pairs together?",
    },
    {
        "ticket_id": "T-105",
        "customer": {"tier": "gold", "orders_last_year": 9},
        "order": {"item": "Sandals", "price_usd": 45, "status": "delivered", "days_since_delivery": 40},
        "message": "The sole came off after one month. Not happy. What can you do?",
    },
    {
        "ticket_id": "T-106",
        "customer": {"tier": "standard", "orders_last_year": 2},
        "order": {"item": "Sneakers", "price_usd": 85, "status": "delivered", "days_since_delivery": 1},
        "message": "I was charged twice for the same order, and the app crashes when I open my orders page.",
    },
]
print(len(TICKETS), "tickets loaded")

6 tickets loaded


In [4]:
def score(states, instructions, levels, temperature=1.0):
    """
    levels  : list from LOW to HIGH. Each level is text, or a dict like {"what": ..., "examples": [...]}
    returns : one dict per state → score, level, confidence, probabilities
    """
    assert 2 <= len(levels) <= 10, "Use 2 to 10 levels (the answer is one digit, 0-9)."
    level_text = "\n".join(
        f"{i}: {lvl if isinstance(lvl, str) else json.dumps(lvl, ensure_ascii=False)}"
        for i, lvl in enumerate(levels)
    )
    task = (f"QUESTION: {instructions}\n"
            f"LEVELS (from low to high):\n{level_text}\n"
            f"Answer with only the level number.")
    digits = {str(i): [str(i)] for i in range(len(levels))}

    results = []
    for row in next_token_logits([build_prompt(s, task) for s in states]):
        probs = answer_probs(row, digits, temperature)
        value = sum(i * p.item() for i, p in enumerate(probs))   # expected level
        results.append({
            "score": round(value, 2),
            "level": round(value),                               # the nearest level, if you need one answer
            "confidence": round(confidence(probs), 2),
            "probabilities": {i: round(p.item(), 3) for i, p in enumerate(probs)},
        })
    return results

In [5]:
# 🔧 CHANGE: levels. Describe SITUATIONS, not just words like "low" or "high".
#   Bad : ["low", "medium", "high"]   → the model has nothing to compare against
#   Good: what each level LOOKS LIKE in a real ticket
URGENCY = {
    "instructions": "How urgent is this ticket for the business?",
    "levels": [
        "No time pressure; a question or feedback",
        "Handle this week; a normal problem with an order",
        "Handle today; the customer is blocked or money is involved",
        "Handle now; legal threat, safety issue, or many customers affected",
    ],
}

# Structured levels: add examples when the model keeps mixing up two levels.
# 🔧 CHANGE: examples should look like YOUR real data. Unrelated examples don't help.
FRUSTRATION = {
    "instructions": "How frustrated is the customer?",
    "levels": [
        {"what": "Calm or happy",                          "examples": ["Hi! Quick question", "Thanks a lot"]},
        {"what": "Annoyed but polite",                     "examples": ["This is not great, please fix it"]},
        {"what": "Angry",                                  "examples": ["This keeps happening!", "Unacceptable"]},
        {"what": "Furious; threatens to leave or report",  "examples": ["I'm done with you", "I'll report this"]},
    ],
}

print(json.dumps(score([TICKETS[1]], URGENCY["instructions"], URGENCY["levels"])[0], indent=2))

{
  "score": 2.67,
  "level": 3,
  "confidence": 0.54,
  "probabilities": {
    "0": 0.0,
    "1": 0.001,
    "2": 0.326,
    "3": 0.673
  }
}


In [6]:
# Mix model scores + facts from your data into ONE priority number.
# 🔧 CHANGE: WEIGHTS → what matters more to YOUR business. Keep the sum at 1.
WEIGHTS = {"urgency": 0.6, "frustration": 0.25, "customer_value": 0.15}

# 🔧 CHANGE: your own customer tiers.
# Don't ask the model what your code already knows!
TIER_VALUE = {"new": 0.3, "standard": 0.5, "gold": 1.0}

urg = score(TICKETS, URGENCY["instructions"], URGENCY["levels"])
fru = score(TICKETS, FRUSTRATION["instructions"], FRUSTRATION["levels"])

rows = []
for t, u, f in zip(TICKETS, urg, fru):
    u_n = u["score"] / (len(URGENCY["levels"]) - 1)        # scale 0..3 → 0..1
    f_n = f["score"] / (len(FRUSTRATION["levels"]) - 1)
    v_n = TIER_VALUE[t["customer"]["tier"]]                 # from your data, not the model
    priority = WEIGHTS["urgency"] * u_n + WEIGHTS["frustration"] * f_n + WEIGHTS["customer_value"] * v_n
    rows.append({
        "ticket_id": t["ticket_id"],
        "urgency": u["score"],
        "urgency_means": URGENCY["levels"][u["level"]],
        "frustration": f["score"],
        "urgency_conf": u["confidence"],   # below about 0.4? Your levels may overlap. Reword them.
        "priority": round(priority, 2),
    })

pd.DataFrame(rows).sort_values("priority", ascending=False)

,ticket_id,urgency,urgency_means,frustration,urgency_conf,priority
1,T-102,2.67,"Handle now; legal threat, safety issue, or man...",2.20,0.54,0.79
4,T-105,2.16,Handle today; the customer is blocked or money...,2.05,0.52,0.75
0,T-101,2.02,Handle today; the customer is blocked or money...,2.04,0.85,0.72
5,T-106,2.16,Handle today; the customer is blocked or money...,2.08,0.63,0.68
2,T-103,2.01,Handle today; the customer is blocked or money...,1.99,0.79,0.64
3,T-104,1.87,Handle today; the customer is blocked or money...,1.71,0.30,0.56


## 🎉 Summary
| Method | Tokens we compare | You get |
|---|---|---|
| Noul | "Yes" vs "No" | P(yes) |
| Choice | "A", "B", "C"... | best option + probabilities + confidence |
| Score | "0", "1", "2"... | expected level + probabilities + confidence |